### Establish connection

In [ ]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

try:
    from dateutil import parser as date_parser
except ImportError:
    date_parser = None

load_dotenv()

True

In [3]:
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    raise ValueError("❌ Missing required environment variables. Check your .env file.")

try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    articles_collection = db[ARTICLES_COLLECTION_NAME]
    client.admin.command("ping")
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")
except Exception as e:
    raise RuntimeError(f"❌ MongoDB Connection Error: {e}")

✅ Connected to MongoDB Atlas! Database: tuone


### General search

In [10]:
offset = 0
limit = 5

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^13"}}, 
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

for article in articles_to_process:
    print(article["title"])

    val = article.get("validation")

    if val is True:
        print("⏭️  Skipping – article is validated")
        continue
    elif isinstance(val, (int, float)):           # timestamp
        processed_on = datetime.fromtimestamp(val, tz=timezone.utc) \
                                 .strftime("%Y‑%m‑%d %H:%M UTC")
        print(f"⏭️  Skipping – article was validated on {processed_on}")
        continue

Ten Most Read News on OffshoreWIND.biz in 2024
⏭️  Skipping – article was validated on 1970‑01‑01 00:00 UTC
Windar Taps PORR to Build Wind Tower Factory in Poland
⏭️  Skipping – article is validated
Offshore Wind Turbines in 2024: 20+ MW Prototypes Rolling Out in Europe, China; 16 MW Remains Most Powerful Turbine Installed Offshore
Ørsted and Eversource Left Stranded in Rhode Island
Dajin Rolls Out First TP-Less Monopile for Denmark’s Largest Offshore Wind Farm


### Validation metrics

In [4]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

Number of validated articles: 168


### Publication date metrics

In [5]:
# Check meta.date consistency
def classify_date(value):
    if isinstance(value, datetime):
        return "bson"
    if isinstance(value, str):
        if date_parser:
            try:
                date_parser.parse(value)
                return "string"
            except (ValueError, OverflowError):
                return "invalid"
    return "invalid"

In [6]:
counts = {"bson": 0, "string": 0, "invalid": 0, "missing": 0}
invalid_ids = []

cursor = articles_collection.find({}, {"_id": 1, "meta.date": 1}, batch_size=10_000)
total = 0

for doc in cursor:
    total += 1
    meta = doc.get("meta", {})
    if "date" not in meta:
        counts["missing"] += 1
        invalid_ids.append(doc["_id"])
        continue
    category = classify_date(meta["date"])
    counts[category] += 1
    if category == "invalid":
        invalid_ids.append(doc["_id"])

In [7]:
print("\n--- meta.date consistency report ---")
print(f"Total documents: {total:,}")
for key in ["bson", "string", "invalid", "missing"]:
    n = counts[key]
    pct = (n / total * 100) if total else 0
    print(f"{key.capitalize():>8}: {n:>7,}  ({pct:5.1f}%)")


--- meta.date consistency report ---
Total documents: 39,932
    Bson:  39,545  ( 99.0%)
  String:      40  (  0.1%)
 Invalid:     347  (  0.9%)
 Missing:       0  (  0.0%)


In [ ]:
# return all date entries
cursor = articles_collection.find(
    {"meta.date": {"$type": "date"}},
    {"_id": 0, "meta.date": 1}
)
dates = [doc["meta"]["date"].date() for doc in tqdm(cursor)]
df = pd.DataFrame(dates, columns=["date"])

daily_counts = df.value_counts().reset_index(name="count").sort_values("date")
daily_counts.columns = ["date", "count"]
daily_counts["date"] = pd.to_datetime(daily_counts["date"])  # ensure correct dtype
daily_counts["year"] = daily_counts["date"].dt.year

output_dir = "descriptive/daily_counts"

# save one plot per year
years = sorted(daily_counts["year"].unique())
for year in years:
    fig, ax = plt.subplots(figsize=(12, 4))
    year_df = daily_counts[daily_counts["year"] == year]
    ax.plot(year_df["date"], year_df["count"], marker="", linewidth=1)
    ax.set_title(f"Daily Article Count – {year}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Articles per Day")
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    fig_path = os.path.join(output_dir, f"daily_count_{year}.png")
    plt.savefig(fig_path)
    plt.close(fig)

print(f"✅ Saved {len(years)} plots to {output_dir}")

39545it [00:00, 46550.58it/s]


✅ Saved 19 plots to descriptive/daily_counts


### Duplicate entries

In [9]:
# return all URLs
cursor = articles_collection.find(
    {"meta.url": {"$exists": True}},  # ensure url field exists
    {"_id": 1, "meta.url": 1}
)

# build dataframe
data = [{"_id": doc["_id"], "url": doc["meta"]["url"]} for doc in tqdm(cursor)]
df_urls = pd.DataFrame(data)

# count duplicate URLs
url_counts = df_urls["url"].value_counts()
duplicates = url_counts[url_counts > 1]

# display result
print(f"🔍 Total articles: {len(df_urls)}")
print(f"❗ Duplicate URLs found: {len(duplicates)}")
print(duplicates.head(40))  # top 10 most duplicated URLs

39932it [00:01, 33032.20it/s]

🔍 Total articles: 39932
❗ Duplicate URLs found: 1405
url
https://www.world-energy.org/tag/150.html                                                                                                8
https://www.world-energy.org/tag/244.html                                                                                                7
https://www.world-energy.org/tag/176.html                                                                                                6
https://www.pv-magazine.com/author/vincent/                                                                                              5
https://www.pv-magazine.com/marktuebersichten/large-scale-storage-systems/                                                               5
https://www.pv-magazine.com/author/annefischer/                                                                                          5
https://www.pv-magazine.com/author/sergiomatalucci/                                                          

In [16]:
# Step 1: Query for all titles
cursor = articles_collection.find(
    {"title": {"$exists": True}},  # only documents with title
    {"_id": 1, "title": 1}
)

# Step 2: Build dataframe
data = [{"_id": doc["_id"], "title": doc["title"]} for doc in tqdm(cursor)]
df_titles = pd.DataFrame(data)

# Step 3: Count duplicate titles
title_counts = df_titles["title"].value_counts()
duplicate_titles = title_counts[title_counts > 1]

# Step 4: Display result
print(f"🔍 Total articles: {len(df_titles)}")
print(f"❗ Duplicate titles found: {len(duplicate_titles)}")
print(duplicate_titles.head(40))

39721it [00:01, 28990.04it/s]

🔍 Total articles: 39721
❗ Duplicate titles found: 1677
title
No Title Found                                                                                        1592
Utility Scale PV                                                                                       601
Modules & Upstream Manufacturing                                                                       412
Hydrogen                                                                                                73
Energy Storage                                                                                          35
Balance of Systems                                                                                      26
The pv magazine weekly news digest                                                                      11
Nuclear Power                                                                                           10
Solar Power Plant                                                                  

In [17]:
# Step 1: Filter only titles that are duplicates
duplicate_title_counts = title_counts[title_counts > 1]

# Step 2: Count how many titles appear 2 times, 3 times, etc.
dup_frequency_distribution = duplicate_title_counts.value_counts().sort_index(ascending=False)

# Step 3: Display the distribution
print("🧮 Distribution of duplicate title frequencies:")
print(dup_frequency_distribution)

🧮 Distribution of duplicate title frequencies:
count
1592       1
601        1
412        1
73         1
35         1
26         1
11         1
10         1
8          3
7          3
6          9
5         15
4         80
3        257
2       1302
Name: count, dtype: int64


### placeholder for further metrics